# Preliminaries

## Import the Necessary Modules

In [1]:
import json
import pandas as pd
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

## Define Main Query Function

In [55]:
async def query(query_string, store = "L"):
    
    # three Swiss triplestores
    if store == "F":
        address = 'https://fedlex.data.admin.ch/sparqlendpoint'
    elif store == "G":
        address = 'https://geo.ld.admin.ch/query'
    else:
        address = 'https://ld.admin.ch/query'
    
    
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "text/csv" })),
        )
    except Exception:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            df = pd.read_csv(StringIO(res))
            return df
    else:
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + resp.status)

## Define Display Function

Displays pandas dataframe resulting from the SPARQL query as HTML with clickable links.

In [9]:
def display_result(df):
    df = HTML(df.to_html(render_links=True, escape=False))
    display(df)

# Tutorial

In [56]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT * WHERE {
	?DefinedTermSet a schema:DefinedTermSet;
        schema:name ?Name.
  	FILTER(regex(str(?DefinedTermSet), "admin.ch" ) )
    FILTER(lang(?Name) = "de")
}

""")

display_result(df)

,DefinedTermSet,Name
0,https://culture.ld.admin.ch/isil,ISIL-Kennzeichen
1,https://register.ld.admin.ch/opendataswiss/category,Kategorien von opendata.swiss
2,https://ld.admin.ch/ech/97/legalforms,Rechtsformen
3,https://ld.admin.ch/dimension/zefix,Zefix - Zentraler Firmenindex
4,https://ld.admin.ch/dimension/department,Eidgenössische Departemente und die Bundeskanzlei
5,https://ld.admin.ch/dimension/office,Bundesämter
6,https://ld.admin.ch/dimension/canton,Kantone
7,https://ld.admin.ch/dimension/municipality,Gemeinden
8,https://ld.admin.ch/cube/dimension/el01,Schwermetall
9,https://register.ld.admin.ch/opendataswiss/org,Die Organisationen von opendata.swiss


In [62]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?Municipality ?Name ?Population WHERE {
    ?Municipality gn:featureCode gn:A.ADM3 .
    ?Municipality schema:name ?Name .
    ?Municipality gn:population ?Population .
    ?Municipality <http://purl.org/dc/terms/issued> ?Date .
    FILTER (?Date = "2022-01-01"^^xsd:date)
}
ORDER BY DESC(?Population)
LIMIT 5

""", "G")

display_result(df)

,Municipality,Name,Population
0,https://geo.ld.admin.ch/boundaries/municipality/261:2022,Zürich,421878
1,https://geo.ld.admin.ch/boundaries/municipality/6621:2022,Genève,203856
2,https://geo.ld.admin.ch/boundaries/municipality/2701:2022,Basel,173863
3,https://geo.ld.admin.ch/boundaries/municipality/5586:2022,Lausanne,140202
4,https://geo.ld.admin.ch/boundaries/municipality/351:2022,Bern,134794


In [61]:
df = await query("""

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?eventLabel ?decisionDate ?sourceId  WHERE {
  ?actAbstract jolux:classifiedByTaxonomyEntry ?concept .
  ?concept skos:notation ?notation .
  filter(datatype(?notation) = <https://fedlex.data.admin.ch/vocabulary/notation-type/id-systematique>)
  filter(str(?notation) = '170.512')

  ?actAbstract  jolux:basicAct ?basicAct .
  ?draft jolux:hasResultingLegalResource ?basicAct .
  ?draft rdf:type jolux:InitialDraft .

  ?draft jolux:draftHasLegislativeTask ?event . 
  ?event jolux:legislativeTaskType ?eventType .
  ?eventType skos:prefLabel ?eventLabel . filter(lang(?eventLabel) = 'fr')
  ?event jolux:decisionDate ?decisionDate .
  optional {
    ?event jolux:legislativeTaskHasResultingLegalResource ?source .
    ?source jolux:isRealizedBy ?expression .
    ?expression jolux:historicalLegalId ?sourceId .
    ?expression jolux:language <http://publications.europa.eu/resource/authority/language/FRA> .
  }
} 
order by desc(?decisionDate)

""", "F")

display_result(df)

,eventLabel,decisionDate,sourceId
0,Expiration du délai référendaire,2004-10-07,NaN
1,Arrêté du Parlement,2004-06-18,FF 2004 2919
2,Message du Conseil fédéral,2003-10-22,FF 2003 7047
3,Arrêté du Parlement,1986-03-21,FF 1986 I 858
4,Message du Conseil fédéral,1983-06-29,FF 1983 III 441
